# Project Setup & Configuration

In [1]:

from pathlib import Path
import sys
import json
import cv2
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image

PROJECT_ROOT = Path.cwd().parent

SRC_DIR = PROJECT_ROOT / "src"
CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints"
DATA_DIR = PROJECT_ROOT / "data"
MODEL_DIR = PROJECT_ROOT / "models"
REGISTERED_FACES_DIR = PROJECT_ROOT / "registered_faces"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

EMBEDDING_SIZE = 128
INPUT_SIZE = 112


# This was derived from test pairs, not validation pairs.
FACE_THRESHOLD = 0.9489252368609133

sys.path.insert(0, str(PROJECT_ROOT))

from src.model import FaceRecognitionModel

# Load Trained Face Recognition Model

In [2]:
CHECKPOINT_PATH = CHECKPOINT_DIR / "latest_checkpoint.pth"

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=DEVICE,
    weights_only=False
)

model_state = checkpoint["model_state_dict"]

num_classes = model_state["arcface.weight"].shape[0]

model = FaceRecognitionModel(
    embedding_size=EMBEDDING_SIZE,
    num_classes=num_classes
).to(DEVICE)

model.load_state_dict(model_state, strict=True)
model.eval()

face_backbone = model.backbone
face_backbone.eval()

identity_to_class = checkpoint["identity_to_class"]

print(f"Model loaded | Classes: {num_classes} | Device: {DEVICE}")

Model loaded | Classes: 5103 | Device: cuda


# RetinaFace Detection and Alignment

In [3]:
from retinaface import RetinaFace

def detect_and_align_faces(image):

    detections = RetinaFace.detect_faces(image)

    if not detections:
        return [], []

    aligned_faces = RetinaFace.extract_faces(
        img_path=image,
        align=True,
        target_size=(INPUT_SIZE, INPUT_SIZE)
    )

    face_detections = list(detections.values())

    return face_detections, aligned_faces


def get_single_aligned_face(image):

    detections, aligned_faces = detect_and_align_faces(image)

    if len(detections) == 0:
        return None, "No face detected"

    if len(detections) > 1:
        return None, "Multiple faces detected"

    if len(aligned_faces) == 0:
        return None, "Face alignment failed"

    return aligned_faces[0], "Face detected and aligned"

# Face Anti-Spoofing (DeepFace FasNet)

In [4]:
from deepface import DeepFace

def check_face_liveness(image):

    results = DeepFace.extract_faces(
        img_path=image,
        detector_backend="retinaface",
        enforce_detection=True,
        align=True,
        anti_spoofing=True
    )

    if len(results) == 0:
        return False, 0.0, "No face detected"

    if len(results) > 1:
        return False, 0.0, "Multiple faces detected"

    is_real = results[0]["is_real"]
    score = results[0]["antispoof_score"]

    if is_real:
        return True, float(score), "Real face detected"

    return False, float(score), "Spoof suspected"

# MobileFaceNet Embedding Extraction

In [5]:
def extract_face_embedding(aligned_face):

    face = Image.fromarray(
        np.uint8(np.clip(aligned_face, 0, 255))
    ).convert("RGB")

    face = face.resize((INPUT_SIZE, INPUT_SIZE))

    face = np.asarray(face, dtype=np.float32) / 255.0

    # Normalize pixel values from [0, 1] to [-1, 1]
    face = (face - 0.5) / 0.5

    face_tensor = torch.from_numpy(face)
    face_tensor = face_tensor.permute(2, 0, 1).unsqueeze(0)
    face_tensor = face_tensor.to(DEVICE)

    with torch.inference_mode():
        embedding = face_backbone(face_tensor)

        embedding = F.normalize(embedding, p=2, dim=1)

    return embedding.squeeze(0).cpu().numpy()

In [6]:
dummy_face = np.random.randint(
    0, 256, (112, 112, 3), dtype=np.uint8
)

embedding = extract_face_embedding(dummy_face)

print("Embedding shape:", embedding.shape)

Embedding shape: (128,)


# Register New User Embeddings

In [7]:
import uuid
from datetime import datetime

REGISTERED_FACES_DIR.mkdir(parents=True, exist_ok=True)

def register_user(name, aligned_faces):

    if len(aligned_faces) == 0:
        raise ValueError("No face images provided.")

    user_id = str(uuid.uuid4())

    embeddings = []

    for face in aligned_faces:
        embedding = extract_face_embedding(face)
        embeddings.append(embedding)

    embeddings = np.stack(embeddings)

    user_data = {
        "user_id": user_id,
        "name": name,
        "registered_at": datetime.now().isoformat(),
        "embeddings": embeddings
    }

    save_path = REGISTERED_FACES_DIR / f"{user_id}.npz"

    np.savez_compressed(
        save_path,
        user_id=user_id,
        name=name,
        registered_at=user_data["registered_at"],
        embeddings=embeddings
    )

    return user_id

# User Database and Authentication Matching

In [9]:
def load_registered_users():
    users = []

    for file_path in REGISTERED_FACES_DIR.glob("*.npz"):
        data = np.load(file_path, allow_pickle=False)

        users.append({
            "user_id": str(data["user_id"]),
            "name": str(data["name"]),
            "embeddings": data["embeddings"].astype(np.float32)
        })

        data.close()

    return users


def authenticate_embedding(query_embedding):

    users = load_registered_users()

    if not users:
        return None, 0.0, "No registered users"

    query_embedding = F.normalize(
        torch.tensor(query_embedding, dtype=torch.float32),
        p=2,
        dim=0
    ).numpy()

    best_user = None
    best_similarity = -1.0

    for user in users:
        templates = F.normalize(
            torch.tensor(user["embeddings"], dtype=torch.float32),
            p=2,
            dim=1
        ).numpy()

        similarities = templates @ query_embedding
        similarity = float(np.max(similarities))

        if similarity > best_similarity:
            best_similarity = similarity
            best_user = user

    if best_similarity >= FACE_THRESHOLD:
        return best_user, best_similarity, "ACCESS GRANTED"

    return None, best_similarity, "ACCESS DENIED"

# Complete Webcam Authentication Pipeline 

In [10]:
def run_face_authentication():
    camera = cv2.VideoCapture(0)

    if not camera.isOpened():
        print("Camera unavailable")
        return

    print("Press SPACE to authenticate | Q to quit")

    while True:
        ret, frame = camera.read()

        if not ret:
            print("Failed to capture frame")
            break

        display_frame = frame.copy()

        cv2.putText(
            display_frame,
            "SPACE: Authenticate | Q: Quit",
            (20, 35),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (0, 255, 0),
            2
        )

        cv2.imshow("Face Authentication", display_frame)

        key = cv2.waitKey(1) & 0xFF

        if key == ord("q"):
            break

        if key == 32:  # SPACE
            detections, aligned_faces = detect_and_align_faces(frame)

            if len(detections) == 0:
                print("ACCESS DENIED: No face detected")
                continue

            if len(detections) > 1:
                print("ACCESS DENIED: Multiple faces detected")
                continue

            is_real, spoof_score, message = check_face_liveness(frame)

            if not is_real:
                print(f"ACCESS DENIED: {message}")
                continue

            aligned_face = aligned_faces[0]

            embedding = extract_face_embedding(aligned_face)

            user, similarity, result = authenticate_embedding(embedding)

            print(f"\nResult: {result}")
            print(f"Similarity: {similarity:.4f}")

            if user is not None:
                print(f"Welcome, {user['name']}!")

    camera.release()
    cv2.destroyAllWindows()

In [11]:
run_face_authentication()

Press SPACE to authenticate | Q to quit

Result: No registered users
Similarity: 0.0000


In [12]:
def capture_and_register_user(name):
    camera = cv2.VideoCapture(0)

    if not camera.isOpened():
        print("Camera unavailable")
        return

    poses = [
        "Look straight",
        "Turn slightly left",
        "Turn slightly right",
        "Look slightly upward",
        "Look slightly downward",
        "Natural alternate angle"
    ]

    aligned_faces = []
    index = 0

    print("Press SPACE to capture | Q to cancel")

    while index < len(poses):
        ret, frame = camera.read()

        if not ret:
            break

        display = frame.copy()

        cv2.putText(
            display, poses[index],
            (20, 40), cv2.FONT_HERSHEY_SIMPLEX,
            0.8, (0, 255, 0), 2
        )

        cv2.imshow("Face Registration", display)

        key = cv2.waitKey(1) & 0xFF

        if key == ord("q"):
            break

        if key == 32:
            detections, faces = detect_and_align_faces(frame)

            if len(detections) != 1 or len(faces) != 1:
                print("Please ensure exactly one face is visible.")
                continue

            is_real, score, message = check_face_liveness(frame)

            if not is_real:
                print("Capture rejected:", message)
                continue

            aligned_faces.append(faces[0])
            index += 1
            print(f"Captured {index}/6")

    camera.release()
    cv2.destroyAllWindows()

    if len(aligned_faces) == 6:
        user_id = register_user(name, aligned_faces)
        print("Registration successful!")
        print("User ID:", user_id)
    else:
        print("Registration cancelled. No user saved.")

In [13]:
capture_and_register_user("Yuvanesh")

Press SPACE to capture | Q to cancel
Captured 1/6
Captured 2/6
Captured 3/6
Captured 4/6
Captured 5/6
Captured 6/6
Registration successful!
User ID: 50f5ed16-4a47-4d4f-ba89-28b167344f72


In [14]:
run_face_authentication()

Press SPACE to authenticate | Q to quit

Result: ACCESS GRANTED
Similarity: 1.0000
Welcome, Yuvanesh!


In [15]:
def manage_registered_users():
    while True:
        users = load_registered_users()

        print("\n1. View registered users")
        print("2. Search user")
        print("3. Delete user")
        print("4. Exit")

        choice = input("Select option: ")

        if choice == "1":
            for user in users:
                print(user["user_id"], "-", user["name"])

        elif choice == "2":
            name = input("Enter name: ").lower()

            for user in users:
                if name in user["name"].lower():
                    print(user["user_id"], "-", user["name"])

        elif choice == "3":
            user_id = input("Enter user ID to delete: ")
            file_path = REGISTERED_FACES_DIR / f"{user_id}.npz"

            if file_path.exists():
                confirm = input("Confirm deletion? (yes/no): ")

                if confirm.lower() == "yes":
                    file_path.unlink()
                    print("User deleted.")
            else:
                print("User not found.")

        elif choice == "4":
            break

        else:
            print("Invalid option.")

In [16]:
manage_registered_users()


1. View registered users
2. Search user
3. Delete user
4. Exit


Select option:  1


50f5ed16-4a47-4d4f-ba89-28b167344f72 - Yuvanesh

1. View registered users
2. Search user
3. Delete user
4. Exit


Select option:  2
Enter name:  Yuvanesh


50f5ed16-4a47-4d4f-ba89-28b167344f72 - Yuvanesh

1. View registered users
2. Search user
3. Delete user
4. Exit


Select option:  3
Enter user ID to delete:  50f5ed16-4a47-4d4f-ba89-28b167344f72
Confirm deletion? (yes/no):  yes


User deleted.

1. View registered users
2. Search user
3. Delete user
4. Exit


Select option:  1



1. View registered users
2. Search user
3. Delete user
4. Exit


Select option:  4


In [1]:
from pathlib import Path
import os

PROJECT_ROOT = Path.cwd().parent
os.chdir(PROJECT_ROOT)

print("Project:", PROJECT_ROOT)

Project: Y:\Python\AI ML\Month 2\week 8-NLP\45 — APPLE FACE ID  (FACIAL RECOGNITION PROJECTS)


In [ ]:
!streamlit run app.py